# 18 - Multiple Intents and Current Constraints

## Objetivo
Ser honesto sobre uma limitação real e atual da arquitetura: dois intents
não podem compartilhar um mesmo `QuantumRouter` como origem/destino no
mesmo episódio. Mostrar o caso que falha (com uma exceção real, não
simulada), o caso que funciona (nós disjuntos), e documentar - sem
implementar - alternativas arquiteturais para superar essa limitação.

## Conceitos
- `QuantumRouter` hospeda **uma única** `Application` por vez
  (`sequence.topology.node.QuantumRouter`, ver
  `docs/sequence_code_analysis.md`, seção 4.6). `IntentRequestApp` é essa
  aplicação nesta arquitetura.
- Se um segundo intent tentasse reusar o mesmo nó como origem/destino, o
  segundo `IntentRequestApp` sobrescreveria o primeiro no roteador -
  corrompendo silenciosamente os callbacks (`get_reservation_result`,
  `get_memory`) e, com eles, o lifecycle e a contagem de pares entregues
  do primeiro intent.
- `SequenceExecutor._check_no_node_conflict` detecta isso **antes** de
  qualquer simulação e levanta `ValueError` explícito - a corrupção
  silenciosa nunca chega a acontecer.


## Parte A - Caso inválido

Dois intents pedindo entrelaçamento entre os mesmos dois nós (em
direções opostas) - o segundo `deploy` deve falhar antes de tocar o
SeQUeNCe.


In [1]:
from ibqn.demos.topologies import three_node_spec
from ibqn.demos.intents import simple_intent
from ibqn.network.sequence_adapter import SequenceAdapter
from ibqn.network.capabilities import NetworkCapabilities
from ibqn.execution.sequence_executor import SequenceExecutor
from ibqn.intent.repository import IntentRepository
from ibqn.planning.planner import IntentPlanner

SEED = 0
spec_conflict = three_node_spec(stop_time_s=0.1)

intent_conflict_a = simple_intent(intent_id="conflict-a", source="a", destination="b", min_fidelity=0.6, requested_pairs=3, duration=0.05)
intent_conflict_b = simple_intent(intent_id="conflict-b", source="b", destination="a", min_fidelity=0.6, requested_pairs=3, duration=0.05)

adapter = SequenceAdapter(spec_conflict, seed=SEED)
repository = IntentRepository()
executor = SequenceExecutor(adapter, repository)
capabilities = NetworkCapabilities(spec_conflict)
planner = IntentPlanner(capabilities)

executor.deploy(intent_conflict_a, planner.plan(intent_conflict_a))
executor.run()
print(f"intent-conflict-a deployed and running: node 'a' now hosts its IntentRequestApp, status={repository.get(intent_conflict_a.id).lifecycle.status.value}")

try:
    executor.deploy(intent_conflict_b, planner.plan(intent_conflict_b))
    print("NO ERROR (unexpected)")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")


2026-07-11 16:09:28,307 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, seed=0


2026-07-11 16:09:28,310 INFO     ibqn.ibqn.planning.planner intent_id=conflict-a - planned route a->r->b, estimated_fidelity=0.6864, purification=False


2026-07-11 16:09:28,311 INFO     ibqn.ibqn.intent.repository intent_id=conflict-a - intent registered


2026-07-11 16:09:28,312 INFO     ibqn.ibqn.intent.repository intent_id=conflict-a - intent transitioned to VALIDATED (schema validated)


2026-07-11 16:09:28,312 INFO     ibqn.ibqn.intent.repository intent_id=conflict-a - intent transitioned to PLANNING (invoking IntentPlanner)


...

2026-07-11 16:09:28,759 INFO     ibqn.ibqn.intent.repository intent_id=conflict-b - intent transitioned to VALIDATED (schema validated)


2026-07-11 16:09:28,759 INFO     ibqn.ibqn.intent.repository intent_id=conflict-b - intent transitioned to PLANNING (invoking IntentPlanner)


2026-07-11 16:09:28,760 INFO     ibqn.ibqn.intent.repository intent_id=conflict-b - intent transitioned to PLANNED (selected by ShortestHopCountRouting; hop fidelities [0.85, 0.85])


2026-07-11 16:09:28,760 INFO     ibqn.ibqn.intent.repository intent_id=conflict-b - intent transitioned to DEPLOYING (applying route b->r->a and submitting reservation to NetworkManager)


intent-conflict-a deployed and running: node 'a' now hosts its IntentRequestApp, status=ACTIVE
ValueError: cannot deploy intent 'conflict-b': node 'b' already hosts the app for intent 'conflict-a' - a QuantumRouter supports only one application at a time (see docs/sequence_code_analysis.md, section 4.6), so these two intents cannot share this node as a source or destination


In [2]:
from ibqn.intent.models import IntentStatus

assert repository.get(intent_conflict_a.id).lifecycle.status == IntentStatus.ACTIVE, "the first intent must be unaffected by the second's rejection"
assert intent_conflict_b.id in [r.intent.id for r in repository.list()], "the conflicting intent was registered before the conflict was detected"
assert repository.get(intent_conflict_b.id).lifecycle.status == IntentStatus.DEPLOYING, (
    "the conflicting intent is left stuck in DEPLOYING: the ValueError is raised as an exception, "
    "not as a clean FAILED transition - see the interpretation below"
)
print("Confirmed: intent-conflict-a's lifecycle is untouched (still ACTIVE); intent-conflict-b was")
print("registered and reached DEPLOYING, but the conflict was caught before any reservation was attempted.")


Confirmed: intent-conflict-a's lifecycle is untouched (still ACTIVE); intent-conflict-b was
registered and reached DEPLOYING, but the conflict was caught before any reservation was attempted.


## Parte B - Caso suportado

Múltiplos intents em pares de nós **disjuntos** de uma topologia em
estrela (`center` + 4 folhas) - `center` é compartilhado como nó
intermediário de swap por ambos, mas nenhum deles o usa como
origem/destino, então não há conflito.


In [3]:
from ibqn.demos.topologies import star_spec
from ibqn.demos.scenarios import ad_hoc_scenario
from ibqn.experiments.runner import run_scenario

spec_star = star_spec(n_leaves=4, stop_time_s=0.1)

intent_1 = simple_intent(intent_id="intent-1", source="leaf1", destination="leaf2", min_fidelity=0.6, requested_pairs=5, duration=0.05)
intent_2 = simple_intent(intent_id="intent-2", source="leaf3", destination="leaf4", min_fidelity=0.6, requested_pairs=5, duration=0.05)

result = run_scenario(ad_hoc_scenario(spec_star, [intent_1, intent_2], seed=SEED), seed=SEED)
r1 = result.get("intent-1")
r2 = result.get("intent-2")

print(f"intent-1 (leaf1 -> leaf2): {r1.final_status.value}")
print(f"intent-2 (leaf3 -> leaf4): {r2.final_status.value}")


2026-07-11 16:09:28,778 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 5 routers, 4 quantum links, seed=0


2026-07-11 16:09:28,779 INFO     ibqn.ibqn.planning.planner intent_id=intent-1 - planned route leaf1->center->leaf2, estimated_fidelity=0.6864, purification=False


2026-07-11 16:09:28,779 INFO     ibqn.ibqn.intent.repository intent_id=intent-1 - intent registered


2026-07-11 16:09:28,780 INFO     ibqn.ibqn.intent.repository intent_id=intent-1 - intent transitioned to VALIDATED (schema validated)


2026-07-11 16:09:28,780 INFO     ibqn.ibqn.intent.repository intent_id=intent-1 - intent transitioned to PLANNING (invoking IntentPlanner)


...

2026-07-11 16:09:28,786 INFO     ibqn.ibqn.intent.repository intent_id=intent-1 - intent transitioned to ACTIVE (reservation approved)


2026-07-11 16:09:28,787 INFO     ibqn.ibqn.intent.repository intent_id=intent-2 - intent transitioned to ACTIVE (reservation approved)


2026-07-11 16:09:30,346 INFO     ibqn.ibqn.intent.repository intent_id=intent-1 - intent transitioned to SATISFIED (all success conditions met)


2026-07-11 16:09:30,349 INFO     ibqn.ibqn.intent.repository intent_id=intent-2 - intent transitioned to SATISFIED (all success conditions met)


intent-1 (leaf1 -> leaf2): SATISFIED
intent-2 (leaf3 -> leaf4): SATISFIED


In [4]:
from ibqn.demos.tables import evaluation_table

print("intent-1 evaluation:")
display(evaluation_table(r1.evaluation))
print("intent-2 evaluation:")
display(evaluation_table(r2.evaluation))


intent-1 evaluation:


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,5.0,124.000000,True,124 DELIVERY event(s) tagged intent_id='intent-1'
1,average_fidelity,>=,0.6,0.686375,True,124 DELIVERY event(s) tagged intent_id='intent-1'


intent-2 evaluation:


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,5.0,119.000000,True,119 DELIVERY event(s) tagged intent_id='intent-2'
1,average_fidelity,>=,0.6,0.686375,True,119 DELIVERY event(s) tagged intent_id='intent-2'


In [5]:
assert r1.final_status == IntentStatus.SATISFIED
assert r2.final_status == IntentStatus.SATISFIED

delivered_1 = next(c.observed for c in r1.evaluation.condition_results if c.metric == "delivered_pairs")
delivered_2 = next(c.observed for c in r2.evaluation.condition_results if c.metric == "delivered_pairs")
assert delivered_1 >= 5 and delivered_2 >= 5
print(f"intent-1 delivered {delivered_1:.0f} pairs; intent-2 delivered {delivered_2:.0f} pairs - each isolated to its own intent_id.")


intent-1 delivered 124 pairs; intent-2 delivered 119 pairs - each isolated to its own intent_id.


## Parte C - Evolução futura (documentado, não implementado)

Superar a limitação de "um app por roteador" exigiria uma das seguintes
mudanças arquiteturais - nenhuma delas implementada nesta versão:

1. **App multiplexador por roteador**: uma única `Application` instalada
   em cada `QuantumRouter`, que despacha `get_reservation_result`/
   `get_memory` para o `IntentRequestApp` correto de acordo com a
   reserva/memória envolvida, em vez de um `IntentRequestApp` por intent.
2. **Registry interno por `reservation.identity`**: mapear cada
   `Reservation.identity` (já existe no núcleo do SeQUeNCe, ver
   `network_management/reservation.py`) para o `intent_id` correspondente
   num registro central, permitindo que várias reservas no mesmo nó sejam
   distinguidas sem precisar de uma `Application` por reserva.
3. **Dispatcher de callbacks**: uma camada intermediária entre
   `QuantumRouter.receive_message` e os callbacks de aplicação, que
   inspeciona a mensagem/memória para decidir qual handler de intent
   chamar - viável sem alterar `sequence.topology.node.QuantumRouter`.
4. **Múltiplas sessões por aplicação**: estender `IntentRequestApp` para
   gerenciar uma lista de `(reservation, intent_id)` internamente, com
   `get_memory` resolvendo qual intent uma memória entregue pertence antes
   de contar/reportar - a mudança mais próxima da estrutura atual.
5. **Alteração controlada do SeQUeNCe ou adaptador externo**: expor um
   ponto de extensão em `QuantumRouter` (ou envolver `Node.receive_message`
   por composição) que aceite múltiplas aplicações - exigiria a menor
   modificação de código de terceiros possível, e só depois de esgotar as
   opções 1-4 (ver a restrição do projeto contra tocar o núcleo do
   SeQUeNCe).

Nenhuma dessas alternativas é implementada aqui - este notebook documenta
a limitação e as opções, não a resolve.


## Interpretação
A Parte A confirma que a arquitetura prefere falhar alto e explícito
(`ValueError`, antes de qualquer simulação) a permitir uma corrupção
silenciosa de callbacks - uma escolha de design deliberada
(`docs/assurance_design.md`, seção 4). A Parte B confirma que o
isolamento por intent (reservas, eventos `DELIVERY`, métricas por
`intent_id`) continua correto mesmo quando um nó (`center`) é
efetivamente compartilhado como repetidor por múltiplos intents - a
restrição é especificamente sobre nós **origem/destino**, não sobre nós
intermediários.

## Limitações
- Esta é a limitação central deste notebook: **um roteador não pode ser
  origem/destino de mais de um intent simultâneo** nesta versão.
- O intent rejeitado (`conflict-b`) fica registrado no repositório, preso
  em `DEPLOYING` - `_check_no_node_conflict` levanta uma exceção Python
  comum, não uma transição de lifecycle para `FAILED`. Um chamador que
  queira lidar com esse caso graciosamente precisa envolver `deploy()` num
  `try/except ValueError` e decidir o que fazer com o registro parado em
  `DEPLOYING` (não há transição automática para um estado terminal).
- Nenhuma das alternativas da Parte C foi prototipada ou testada - são
  apenas as opções de design mais plausíveis dado o que o núcleo do
  SeQUeNCe já expõe (`Reservation.identity`, arquitetura de mensagens).

## Conclusão
A restrição de "um app por roteador" demonstrada com uma falha real e
explícita, contrastada com um caso real e correto de múltiplos intents
isolados em nós disjuntos, e documentada - sem implementação - como uma
limitação a ser endereçada em versões futuras.
